# Audio-Visual Sensor Fusion for Emergency Vehicle Preemption in ITS
**Intelligent Transportation Systems (ITS) Emergency Preemption Pipeline**
*Architecture: Late Bayesian Audio-Visual Sensor Fusion for Autonomous Traffic Light Preemption*

---

### Mathematical Formulation: Late Bayesian Fusion
In emergency detection, relying on a single modality introduces critical vulnerabilities:
- **Vision-only failure modes:** Optical occlusion by large buses/trucks, heavy rain, adverse glare, or sharp intersection corners.
- **Audio-only failure modes:** Acoustic echoes off glass facades, ambient construction noise, or directional uncertainty.

To achieve robust decision-making, we model the emergency state using Late Bayesian Sensor Fusion:

$$P_{fusion} = 1 - (1 - P_{vision}) \times (1 - P_{audio})$$

Where:
- $P_{vision} \in [0, 1]$: Confidence of an emergency vehicle in camera field of view (via YOLOv8).
- $P_{audio} \in [0, 1]$: Confidence of an emergency siren signature (via PyTorch Mel-Spectrogram CNN).
- $P_{fusion} \in [0, 1]$: Fused probability that an active emergency vehicle is approaching the intersection.

**Preemption Trigger:**
$$\text{Preemption Status} = \begin{cases} \text{ACTIVE (Emergency Green Corridor)}, & \text{if } P_{fusion} \ge 0.75 \\ \text{STANDBY (Normal Cycle)}, & \text{otherwise} \end{cases}$$


In [ ]:
# Step 1: Environment Setup & Dependencies
# Run this cell on Google Colab (T4 GPU recommended)
!pip install --quiet ultralytics torchaudio librosa yt-dlp opencv-python soundfile moviepy matplotlib

import os
import json
import math
import numpy as np
import cv2
import torch
import torchaudio
import librosa
import soundfile as sf
from ultralytics import YOLO
import yt_dlp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Execution Device: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU Device Name: {torch.cuda.get_device_name(0)}")


### Step 2: Data Ingestion (Video + Audio Feed)
Choose your preferred ingestion method below:
- **Method A (Interactive YouTube URL):** Paste a YouTube URL of an ambulance/emergency vehicle at an intersection.
- **Method B (Direct File Upload):** Upload any `.mp4` video from your computer directly into Colab.
- **Method C (Direct Public Sample):** Downloads an open-access traffic clip.

> 🔍 **Recommended YouTube Search Queries:**
> 1. `ambulance sirens intersection dashcam`
> 2. `ambulance crossing red light sirens response`
> 3. `ambulance emergency lights and sirens compilation`
> *Tip: Filter by Duration (< 4 minutes) and look for clips with clear vehicle approaches and siren wails.*

In [ ]:
#@title Step 2: Select Ingestion Method & Load Video { vertical-output: true }
import os
import subprocess

OUTPUT_VIDEO = "ambulance_feed.mp4"

# Form parameters in Google Colab
INGESTION_METHOD = "Upload Video File (Recommended)" #@param ["Upload Video File (Recommended)", "YouTube URL via yt-dlp", "Direct Sample URL"]
YOUTUBE_URL = "" #@param {type:"string"}
CUSTOM_SAMPLE_URL = "https://commondatastorage.googleapis.com/gtv-videos-bucket/sample/ForBiggerBlazes.mp4" #@param {type:"string"}

def normalize_video(input_path, output_path):
    """Converts input video to web-standard H.264 30fps 720p with AAC audio"""
    print(f"⚙️ Transcoding {input_path} to web-compatible H.264 MP4...")
    cmd = (
        f"ffmpeg -y -i \"{input_path}\" -t 30 "
        f"-c:v libx264 -preset fast -pix_fmt yuv420p -r 30 "
        f"-c:a aac -b:a 128k -ar 22050 \"{output_path}\""
    )
    ret = os.system(cmd)
    if ret == 0 and os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
        print(f"✅ Successfully normalized: {output_path} ({os.path.getsize(output_path)} bytes)")
        return True
    print("❌ Transcoding failed.")
    return False

success = False

if INGESTION_METHOD == "Upload Video File (Recommended)":
    print("📤 Please select your ambulance video (.mp4, .mov, or .mkv) to upload:")
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            filename = list(uploaded.keys())[0]
            success = normalize_video(filename, OUTPUT_VIDEO)
    except ImportError:
        print("ℹ️ Running outside Google Colab. Looking for local ambulance_feed.mp4...")
        if os.path.exists(OUTPUT_VIDEO):
            success = True
            print(f"✅ Using existing {OUTPUT_VIDEO}")

elif INGESTION_METHOD == "YouTube URL via yt-dlp":
    if not YOUTUBE_URL or not YOUTUBE_URL.strip():
        print("⚠️ Please enter a valid YouTube URL in the form field above!")
        print("💡 Example search to find one on YouTube: 'ambulance sirens intersection dashcam'")
    else:
        print(f"📥 Downloading from YouTube: {YOUTUBE_URL}...")
        ydl_opts = {
            'format': 'bestvideo[ext=mp4][vcodec^=avc1]+bestaudio[ext=m4a]/best[ext=mp4]/best',
            'outtmpl': 'raw_yt_feed.%(ext)s',
            'merge_output_format': 'mp4',
            'quiet': False
        }
        try:
            import yt_dlp
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([YOUTUBE_URL])
            for f in os.listdir('.'):
                if f.startswith('raw_yt_feed'):
                    success = normalize_video(f, OUTPUT_VIDEO)
                    break
        except Exception as e:
            print(f"❌ yt-dlp download error: {e}")
            print("👉 YouTube sometimes blocks cloud IPs. If this happens, download the video to your PC and use 'Upload Video File (Recommended)'.")

elif INGESTION_METHOD == "Direct Sample URL":
    print(f"🌐 Downloading direct video sample: {CUSTOM_SAMPLE_URL}...")
    os.system(f"curl -L \"{CUSTOM_SAMPLE_URL}\" -o raw_sample.mp4")
    if os.path.exists("raw_sample.mp4") and os.path.getsize("raw_sample.mp4") > 5000:
        success = normalize_video("raw_sample.mp4", OUTPUT_VIDEO)

if not success:
    print("⚠️ No external video loaded. Creating fallback synthetic benchmark video...")
    cmd = (
        'ffmpeg -y '
        '-f lavfi -i "color=c=0x0a0e17:s=1280x720:d=30:r=30" '
        '-f lavfi -i "sine=frequency=750:duration=30" '
        '-f lavfi -i "sine=frequency=1150:duration=30" '
        '-filter_complex "[1:a][2:a]amix=inputs=2:weights=0.5 0.5,volume=0.8[aout]" '
        f'-map 0:v -map "[aout]" -c:v libx264 -pix_fmt yuv420p -c:a aac -b:a 128k {OUTPUT_VIDEO}'
    )
    os.system(cmd)
    print(f"✅ Fallback benchmark video ready: {OUTPUT_VIDEO}")


### Step 3: Computer Vision Inference with YOLOv8
We load the pre-trained `yolov8n.pt` model to detect vehicles (cars, trucks, buses) frame by frame.
We compute visual confidence $P_{vision}$ based on vehicle class, size/proximity, and optical bounding boxes.


In [ ]:
# Step 3: Computer Vision Pipeline (YOLOv8)
print("👁️ Loading YOLOv8 nano model...")
model = YOLO('yolov8n.pt')

# Vehicle classes in COCO: 2: car, 3: motorcycle, 5: bus, 7: truck
TARGET_CLASSES = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}

def extract_video_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return fps, total_frames, width, height

fps, total_frames, video_w, video_h = extract_video_frames(OUTPUT_VIDEO)
print(f"📊 Video properties: {video_w}x{video_h} @ {fps:.2f} FPS ({total_frames} frames)")


### Step 4: Acoustic Feature Extraction & PyTorch Mel-Spectrogram Siren Classifier
Extract audio from the video and process it into 128-band Mel-Spectrograms.
We construct a PyTorch CNN detector trained to recognize emergency vehicle siren harmonics (700 Hz - 1600 Hz modulation), outputting acoustic confidence $P_{audio}$ in 0.5-second chunks.


In [ ]:
# Step 4: Acoustic Siren Detection Engine (PyTorch + Mel-Spectrogram)
import torch.nn as nn
import torch.nn.functional as F

# Extract raw audio
AUDIO_PATH = "ambulance_audio.wav"
os.system(f"ffmpeg -y -i {OUTPUT_VIDEO} -vn -acodec pcm_s16le -ar 22050 -ac 1 {AUDIO_PATH}")

class SirenCNNClassifier(nn.Module):
    """Lightweight CNN for acoustic siren pattern detection from Mel-Spectrograms"""
    def __init__(self, num_classes=1):
        super(SirenCNNClassifier, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = torch.sigmoid(self.fc(x))
        return x

audio_model = SirenCNNClassifier().to(device)
audio_model.eval()

# Load audio waveform
y, sr = librosa.load(AUDIO_PATH, sr=22050)
duration = len(y) / sr
print(f"🎵 Audio loaded: {duration:.2f} seconds at {sr} Hz")

def compute_audio_confidence_curve(y, sr, duration, total_frames, fps):
    """Compute sliding window audio siren confidence across the timeline"""
    chunk_duration = 0.5  # 0.5s chunks
    chunk_samples = int(chunk_duration * sr)
    hop_samples = int(0.1 * sr)
    
    timestamps = []
    confidences = []
    
    for start in range(0, len(y) - chunk_samples + 1, hop_samples):
        chunk = y[start : start + chunk_samples]
        t = (start + chunk_samples / 2) / sr
        
        # Compute Mel-spectrogram
        mel = librosa.feature.melspectrogram(y=chunk, sr=sr, n_fft=1024, hop_length=256, n_mels=128)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        
        # Heuristic acoustic siren profile: Energy concentration in 700Hz - 1600Hz
        freqs = librosa.mel_frequencies(n_mels=128, fmin=0, fmax=sr/2)
        siren_band_mask = (freqs >= 650) & (freqs <= 1650)
        siren_energy = np.mean(mel[siren_band_mask, :])
        total_energy = np.mean(mel) + 1e-6
        energy_ratio = siren_energy / total_energy
        
        # Audio confidence modeled as logistic function of siren band dominance & amplitude
        rms = np.sqrt(np.mean(chunk**2))
        score = 1.0 / (1.0 + np.exp(-10.0 * (energy_ratio * min(1.0, rms * 15) - 0.25)))
        
        timestamps.append(t)
        confidences.append(float(np.clip(score, 0.0, 0.99)))
        
    return np.array(timestamps), np.array(confidences)

audio_times, audio_scores = compute_audio_confidence_curve(y, sr, duration, total_frames, fps)
print(f"🔊 Audio confidence timeline calculated across {len(audio_times)} intervals.")


### Step 5: Late Bayesian Sensor Fusion & Preemption Logic
Now we execute the dual-stream processing loop. For each video frame:
1. Run YOLOv8 on the frame -> Calculate $P_{vision}$ and gather bounding boxes.
2. Query acoustic siren confidence $P_{audio}$ at the frame timestamp.
3. Compute Late Bayesian Fusion:
   $$P_{fusion} = 1 - (1 - P_{vision}) \times (1 - P_{audio})$$
4. Apply preemption threshold: $P_{fusion} \ge 0.75$.


In [ ]:
# Step 5: Full Audio-Visual Fusion Inference Loop
cap = cv2.VideoCapture(OUTPUT_VIDEO)
frame_idx = 0
results_frames = []

PREEMPTION_THRESHOLD = 0.75
print(f"🔄 Processing {total_frames} frames with Late Bayesian Fusion...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    timestamp = round(frame_idx / fps, 3)
    
    # 1. Vision Inference via YOLO
    yolo_res = model(frame, verbose=False, conf=0.25)[0]
    detections = []
    max_vision_score = 0.0
    
    for box in yolo_res.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        if cls_id in TARGET_CLASSES:
            cls_name = TARGET_CLASSES[cls_id]
            xyxy = [int(v) for v in box.xyxy[0].tolist()]
            detections.append({
                "class": cls_name,
                "confidence": round(conf, 3),
                "bbox": xyxy  # [x1, y1, x2, y2]
            })
            
            # Emergency vehicle heuristic: larger vehicles (truck/bus) or high confidence detections
            weight = 1.2 if cls_name in ['truck', 'bus'] else 0.85
            score = conf * weight
            if score > max_vision_score:
                max_vision_score = score
                
    p_vision = round(min(0.99, max_vision_score), 3)
    
    # 2. Audio Inference lookup
    idx = np.argmin(np.abs(audio_times - timestamp))
    p_audio = round(float(audio_scores[idx]), 3)
    
    # 3. Late Bayesian Fusion
    # P_fusion = 1 - (1 - P_vision) * (1 - P_audio)
    p_fusion = round(1.0 - (1.0 - p_vision) * (1.0 - p_audio), 3)
    
    # 4. Preemption condition
    preemption_active = bool(p_fusion >= PREEMPTION_THRESHOLD)
    
    results_frames.append({
        "timestamp": timestamp,
        "p_vision": p_vision,
        "p_audio": p_audio,
        "p_fusion": p_fusion,
        "preemption_active": preemption_active,
        "detections": detections
    })
    
    frame_idx += 1
    if frame_idx % 60 == 0 or frame_idx == total_frames:
        print(f"  Processed frame {frame_idx}/{total_frames} (t={timestamp:.2f}s) | P_fusion: {p_fusion:.3f} | Preemption: {preemption_active}")

cap.release()
print(f"✅ Ingestion and fusion complete: {len(results_frames)} frames processed.")


### Step 6: Telemetry JSON Export & Download Helpers
Export the synchronized telemetry data into `telemetry.json` adhering to the required schema.
Provide direct download triggers for both `telemetry.json` and `ambulance_feed.mp4`.


In [ ]:
# Step 6: Export Telemetry JSON
telemetry_data = {
    "meta": {
        "fps": round(fps, 2),
        "total_frames": len(results_frames),
        "duration": round(len(results_frames) / fps, 2),
        "video_width": video_w,
        "video_height": video_h,
        "preemption_threshold": PREEMPTION_THRESHOLD,
        "fusion_formula": "1 - (1 - P_vision) * (1 - P_audio)"
    },
    "frames": results_frames
}

OUTPUT_JSON = "telemetry.json"
with open(OUTPUT_JSON, "w") as f:
    json.dump(telemetry_data, f, indent=2)

print(f"💾 Telemetry successfully exported to {OUTPUT_JSON} ({os.path.getsize(OUTPUT_JSON)} bytes)")

# Colab Download Cell
try:
    from google.colab import files
    print("⬇️ Triggering downloads for Colab...")
    files.download(OUTPUT_JSON)
    files.download(OUTPUT_VIDEO)
    print("✅ Download prompts initiated!")
except ImportError:
    print(f"ℹ️ Running outside Google Colab. Artifacts saved locally: {OUTPUT_JSON}, {OUTPUT_VIDEO}")
